# Inpainting submarino -> tokens DINOv3-VQ (Kaggle)
Requisitos: GPU (T4/P100), Internet ON, secret `HF_TOKEN` (acceso aprobado a `facebook/dinov3-vitb16-pretrain-lvd1689m`),
y un Dataset con: `common.py train.py predict.py dino_vq.py codebook.npy target.csv train.zip test.zip` (o carpetas `train/` `test/`).
Ejecutar con **Save & Run All** (commit en segundo plano).

In [ ]:
import os, glob, subprocess, shutil, sys
# --- Config ---
SMOKE = False            # True: prueba rápida (60 pasos) antes del run largo
TIME_BUDGET_H = 9.5      # el commit de Kaggle dura máx ~12h; deja margen para predict
RESUME_FROM = ""         # p.ej. /kaggle/input/<ckpt-dataset>/last.pt para continuar un entrenamiento
# --- Localizar dataset ---
hits = glob.glob('/kaggle/input/**/target.csv', recursive=True)
assert hits, 'No encuentro target.csv en /kaggle/input'
SRC = os.path.dirname(hits[0]); print('SRC =', SRC)
W = '/kaggle/working'; T = '/kaggle/temp'; os.makedirs(T, exist_ok=True)
for f in ['common.py','train.py','predict.py','dino_vq.py']:
    shutil.copy(glob.glob(f'/kaggle/input/**/{f}', recursive=True)[0], W)
shutil.copy(f'{SRC}/codebook.npy', W); shutil.copy(f'{SRC}/target.csv', W)
os.chdir(W)

In [ ]:
def find_dir(name):
    d = f'{SRC}/{name}'
    if os.path.isdir(d): return d
    z = f'{SRC}/{name}.zip'
    if os.path.exists(z):
        subprocess.run(['unzip','-q','-o',z,'-d',f'{T}/{name}_x'], check=True)
        sub = f'{T}/{name}_x/{name}'
        return sub if os.path.isdir(sub) else f'{T}/{name}_x'
    raise FileNotFoundError(name)
TRAIN, TEST = find_dir('train'), find_dir('test')
print(TRAIN, len(os.listdir(TRAIN)), '|', TEST, len(os.listdir(TEST)))

In [ ]:
!pip install -q -U transformers
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret('HF_TOKEN'))
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
cmd = f"python -u train.py --train-dir {TRAIN} --codebook codebook.npy --out runs/v1 --cache {T}/train.npy"
if SMOKE: cmd += " --max-steps 60 --val-n 50 --val-every 30 --time-budget-h 0.2"
else:     cmd += f" --time-budget-h {TIME_BUDGET_H}"
if RESUME_FROM: cmd += f" --resume {RESUME_FROM}"
!{cmd}

In [ ]:
!python predict.py --ckpt runs/v1/last.pt --test {TEST} --target target.csv --out generate --tta

In [ ]:
!python dino_vq.py --images-root generate --target-csv target.csv --codebook-path codebook.npy --output-csv submission.csv
import pandas as pd
s = pd.read_csv('submission.csv'); print(s.shape, s.Target.min(), s.Target.max(), s.Id.duplicated().sum())

In [ ]:
# Opcional: liberar espacio de salida (solo quedan last.pt y submission.csv)
shutil.rmtree('generate', ignore_errors=True)